# ForecastEx

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Get live markets

In [3]:
from get_live_markets import get_live_markets
markets = get_live_markets()

In [4]:
from add_category_to_markets import add_category_to_markets
df_markets = add_category_to_markets(markets)

In [5]:
df_markets = df_markets[['category', 'name', 'symbol', 'conid']].sort_values(by=['category', 'name'])

In [6]:
conid_to_market = {market['conid']: market for market in markets}

## Get forecastex markets excluding financials

In [7]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [ ]:
for market in tqdm(markets):
    if 'contracts' not in market:
        market['contracts'] = fetch_all_contracts(market['conid'])
        print(market['conid'], len(market['contracts']))

In [24]:
forecastex = [market for market in markets if len(market['contracts']) > 0]

In [26]:
len(markets), len(forecastex)

(204, 65)

## Get current probability for each market contract

In [28]:
from get_candidate_probability import get_candidate_probability

In [33]:
from tqdm.auto import tqdm

In [38]:
for market in tqdm(forecastex):
    for contract in market['contracts']:
        conid = contract['conid']
        if 'probability' not in contract:
            try:
                contract['probability'] = get_candidate_probability(conid)
            except:
                contract['probability'] = None
                print('error', contract['shortDescription'])
        try:
            print(conid, contract['shortDescription'], contract['probability']['probability_pct'])
        except:
            pass

  0%|          | 0/65 [00:00<?, ?it/s]

796056496 MNYCG Nov04'25 Sliwa YES @FORECASTX 4.0
796056501 MNYCG Nov04'25 Sliwa NO @FORECASTX 97.0
796056506 MNYCG Nov04'25 Cuomo YES @FORECASTX 17.0
796056511 MNYCG Nov04'25 Cuomo NO @FORECASTX 84.0
796056520 MNYCG Nov04'25 Mamdani YES @FORECASTX 68.0
796056525 MNYCG Nov04'25 Mamdani NO @FORECASTX 33.0
796056531 MNYCG Nov04'25 Adams YES @FORECASTX 8.0
796056534 MNYCG Nov04'25 Adams NO @FORECASTX 93.0
791103640 FFDEC Jul30'25 Lower25bps YES @FORECASTX 7.000000000000001
791103645 FFDEC Jul30'25 Lower25bps NO @FORECASTX 94.0
791103649 FFDEC Jul30'25 Nochange YES @FORECASTX 94.0
791103652 FFDEC Jul30'25 Nochange NO @FORECASTX 7.000000000000001
791103682 FFDEC Sep17'25 Lower25bps YES @FORECASTX 42.0
791103687 FFDEC Sep17'25 Lower25bps NO @FORECASTX 59.0
791103688 FFDEC Sep17'25 Nochange YES @FORECASTX 46.0
791103693 FFDEC Sep17'25 Nochange NO @FORECASTX 55.00000000000001
791103699 FFDEC Sep17'25 Raise50bpsormore YES @FORECASTX 11.0
791103702 FFDEC Sep17'25 Raise50bpsormore NO @FORECASTX 9

In [46]:
f2 = [x.copy() for x in forecastex]

In [50]:
for market in f2:
    market['contracts'] = [x for x in market['contracts'] if x['probability']]

In [51]:
f2 = [market for market in f2 if market['contracts']]

In [52]:
len(f2), len(forecastex)

(24, 65)

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [54]:
from OI_to_integer import OI_to_integer

In [ ]:
for market in tqdm(f2):
    for contract in tqdm(market['contracts']):
        if 'OI' not in contract:
            conid = contract['conid']
            OI = await run_get_OI([conid])

## Collect and filter

In [200]:
rows = []
for market in f2:
    for ctx in market['contracts']:
        contract = ctx.copy()
        probability = contract['probability']
        contract['probability_pct'] = probability['probability_pct']
        contract['weekly_volume'] = sum(probability['volume'])
        contract['yesNo'] = 'YES' if contract['putOrCall'] == 'C' else 'NO'
        for x in ['putOrCall','popularityRank', 'expiration', 'lastTradeMillis', 'lastTradeTime','eventFixedPayout', 'commodityCode', 'strike', 'market',
                    'categories', 'expectedResolutionTime', 'expectedPayoutTime', 'timespecifierParam', 'sourceAgency', 'marketRulesLink', 
                    'exchange', 'priceIncrement', 'currency', 'timezone', 'eventAuthorityURL', 'probability', 'tradingHours']:
            del contract[x]
        try:
            OI = contract['OI']
            OI = list(OI.values())[0]
            contract['OI'] = OI_to_integer(OI)
        except:
            pass
        rows.append(contract)

from add_category_to_markets import add_category_to_markets
df = add_category_to_markets(rows)
df = df[['category','underlyingName', 'longDescription', 'shortDescription', 'strikeLabel', 'yesNo', 'probability_pct',  'OI', 'lastTradeDate', 'conid']].sort_values(by=['category', 'underlyingName', 'strikeLabel', 'yesNo']).reset_index(drop=True)
df1 = df[(df.probability_pct > 25 ) & (df.probability_pct < 75 ) & (df.yesNo == 'YES')].reset_index(drop=True)

In [201]:
df1.to_csv('forecastex.csv')

In [213]:
df2 = df1.sort_values(by=['underlyingName', 'strikeLabel'])
df2 = df2[['underlyingName', 'longDescription', 'shortDescription', 'strikeLabel', 'yesNo', 'probability_pct', 'OI', 'lastTradeDate', 'conid']]
df2 = df2[df2.lastTradeDate <= '20251028']
df2.sort_values(by='lastTradeDate')

,underlyingName,longDescription,shortDescription,strikeLabel,yesNo,probability_pct,OI,lastTradeDate,conid
5,Mayoral Democratic Primary in Seattle,Will Bruce Harrell win the Seattle Democratic Primary election for Mayor in 2025?,MSEAD Aug05'25 Harrell YES @FORECASTX,Harrell,YES,62.0,1740.0,20250819,771327516
17,Fed Decision,"Will the Fed lower the rate 25 bps at the September 17, 2025 meeting?",FFDEC Sep17'25 Lower25bps YES @FORECASTX,Lower 25bps,YES,42.0,300.0,20250917,791103682
19,Fed Decision,"Will the Fed leave the rate unchanged at the September 17, 2025 meeting?",FFDEC Sep17'25 Nochange YES @FORECASTX,No change,YES,46.0,461.0,20250917,791103688
21,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Automobiles and Auto Parts by September 30, 2025?",USTRF Sep30'25 Automobiles YES @FORECASTX,Automobiles,YES,30.0,10.0,20251001,788715383
23,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on the European Union by September 30, 2025?",USTRF Sep30'25 EuropeanUnion YES @FORECASTX,EuropeanUnion,YES,31.0,10.0,20251001,788715389
24,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Russia by September 30, 2025?",USTRF Sep30'25 Russia YES @FORECASTX,Russia,YES,44.0,20.0,20251001,788715372
25,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Semiconductors by September 30, 2025?",USTRF Sep30'25 Semiconductors YES @FORECASTX,Semiconductors,YES,35.0,60.0,20251001,788715405
27,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Smartphones by September 30, 2025?",USTRF Sep30'25 Smartphones YES @FORECASTX,Smartphones,YES,36.0,20.0,20251001,788715420


In [204]:
from datetime import datetime

In [210]:
ifps[0]['props']

{'title': "Will any of the EU or G7 countries decide to confiscate the Russian central bank's already frozen assets by the end of November 2025?",
 'ai_title': '',
 'shortTitle': 'Confiscation of frozen Russian assets?',
 'details': '<b>Background:</b> Countries belonging to the EU and/or the G7 are estimated to have frozen around €300 billion worth of assets belonging to the Russian central bank. The majority of the assets are managed by Euroclear in Belgium. The EU and G7 countries have declared that these assets will remain frozen until Russia pays for the damage it has caused to Ukraine. There have long been discussions about confiscating these assets and instead using them to support Ukraine.\n<p><b>Resolution:</b> The question will be decided on the basis of reports from credible news sources confirmed by official statements from the country or countries concerned. For the question to be resolved as “Yes”, at least one EU and/or G7 country will have to decide on the confiscation 

In [221]:
def make_glimt_question(row):
    return {'id': row.conid,
            'type': 'bins',
            'state': 'active',
            'dates': {'startDay': str(datetime.now())[0:10].replace('-',''),'endDay': row.lastTradeDate},
            'props': {'title': row.longDescription, 'shortTitle': row.shortDescription, 'details': ''},
            'kind': 'discrete',
            'bins': [{'props': {'title': 'Yes', 'ai_title': '', 'order': 0, 'color': ''},
                        'isActive': True,
                        'endDay': 0},
                    {'props': {'title': 'No', 'ai_title': '', 'order': 0, 'color': ''},
                        'isActive': True,
                        'endDay': 0}]}

In [222]:
ifps = df2.apply(make_glimt_question, axis=1)

In [223]:
len(ifps)

8

In [224]:
import os
os.makedirs('glimt/prompt', exist_ok=True)

In [225]:
from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

In [226]:
from gather_news_for_ifps import gather_news_for_ifps
news = gather_news_for_ifps(ifps)

saved glimt/news/791103682.txt
saved glimt/news/791103688.txt
saved glimt/news/771327516.txt
saved glimt/news/788715383.txt
saved glimt/news/788715389.txt
saved glimt/news/788715372.txt
saved glimt/news/788715405.txt
saved glimt/news/788715420.txt


In [227]:
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from humor_me import humor_me
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from datetime import datetime

loading massive wiki index 2025-07-24 21:51:33.210260
loading wiki article titles 2025-07-24 21:52:41.002267
loading sentence transformer model 2025-07-24 21:52:44.181085
done 2025-07-24 21:52:49.595186


In [ ]:
for ifp in ifps[1:]:
    print('begin FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())
    title_plus_criteria = detailed_proposition(ifp)
    wiki_articles = wiki_semantic_search(title_plus_criteria)
    ifp_news_sources, ifp_news_text = split_news_into_text_and_urls(ifp, news)
    source_summaries, sources = create_source_summaries(ifp['id'], title_plus_criteria, wiki_articles, ifp_news_sources, ifp_news_text)
    rephrase_binary_outcomes(ifp)
    research = format_research(source_summaries)
    prompt, rejected = glimt_forecast_prompt(ifp, research)
    ## Run the prompt 5 times
    prompt_tries = 2 # Waste of time on Mistral 4 bit
    answers = [humor_me(prompt, i+1) for i in range(prompt_tries)]
    binProbs = [get_bin_probs(a) for a in answers]
    rights = [get_rights(a) for a in answers]
    wrongs = [get_wrongs(a) for a in answers]
    ## Median forecasts and rationales
    forecast = rejected + median_forecast(binProbs)
    right = median_rationale(rights)
    wrong = median_rationale(wrongs)
    result = (forecast, right, wrong, sources)
    fn = f'glimt/forecast'
    import os
    os.makedirs(fn, exist_ok=True)
    fn = f"{fn}/{ifp['id']}.json"
    import json
    with open(fn, 'w') as f:
        json.dump(result, f)


In [253]:
ifp

{'id': 788715420,
 'type': 'bins',
 'state': 'active',
 'dates': {'startDay': '20250724', 'endDay': '20251001'},
 'props': {'title': 'Will the United States impose additional tariffs on Smartphones by September 30, 2025?',
  'shortTitle': "USTRF Sep30'25 Smartphones YES @FORECASTX",
  'details': ''},
 'kind': 'discrete',
 'bins': [{'props': {'title': 'The United States will impose additional tariffs on smartphones by September 30, 2025.',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'The United States will not impose additional tariffs on smartphones by September 30, 2025.',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0}]}

In [262]:
contracts = df.to_dict(orient='records')

In [266]:
contracts = {contract['conid']: contract for contract in contracts}

In [269]:
for ifp in ifps:
    fn = f'glimt/forecast'
    fn = f"{fn}/{ifp['id']}.json"
    with open(fn, 'r') as f:
        forecast = json.load(f)
    [[Pyes, Pno], Rfor, Ragainst, sources] = forecast
    contract = contracts[ifp['id']]
    yesNo = contract['yesNo']
    pct = contract['probability_pct']
    yesPct = pct if yesNo == 'YES' else 100-pct
    noPct = 100-pct if yesNo == 'YES' else pct
    report = f"""
{ifp['props']['title']}
{ifp['props']['shortTitle']}

YES {100*Pyes} bot / {yesPct} crowd
NO  {100*Pno} bot  / {noPct} crowd

Reasons for YES
===============
{Rfor}

Reasosn for NO
==============
{Ragainst}

----------------------------------------"""
    print(report)


Will the Fed lower the rate 25 bps at the September 17, 2025 meeting?
FFDEC Sep17'25 Lower25bps YES @FORECASTX

YES 60.0 bot / 42.0 crowd
NO  40.0 bot  / 58.0 crowd

Reasons for YES
The Federal Reserve's decisions on interest rates are typically guided by economic conditions, aiming to balance inflation control and economic growth. By September 2025, inflation may stabilize, and economic expansion could moderate, prompting the Fed to consider rate cuts to maintain stability. Historical patterns, including past supply chain disruptions and geopolitical uncertainties, suggest the Fed may prioritize economic stability. Additionally, market expectations and investor sentiment often influence the Fed's actions, as aligning with broader consensus can reinforce economic confidence. The Fed's dual mandate—promoting maximum employment and stable prices—further shapes its approach, ensuring decisions support long-term economic health.

Reasosn for NO
Economic forecasts may be affected by variou